# 1. Libraries

In [ ]:
import dill
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, norm
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# 2. Gather Files

In [3]:
filename = 'dataset_unique_COMBINED_install_1990_cement_3_exposure_2.pkl'

with open(filename, 'rb') as f:
    combined_df_unique = dill.load(f)

print(f"Dataset has been loaded with dimensions: {combined_df_unique.shape}")

# 2. Separating features and targets
features = ['fck', 'rh', 'cov', 'time']
targets  = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']

X = combined_df_unique[features].to_numpy()
y = combined_df_unique[targets].to_numpy()
print("Data separated into X (features) and y (targets).")

Dataset has been loaded with dimensions: (400, 8)
Data separated into X (features) and y (targets).


# 3. Train model

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training with {len(X_train)} samples (80%)")

# 2. Neural Network Configuration
mlp_config = {
    'hidden_layer_sizes': (64, 64),
    'max_iter': 2000,
    'random_state': 42,
    'early_stopping': True,
    'validation_fraction': 0.20 # Internal validation automatically set to 20% of X_train
}

# 3. Training 4 Independent Neural Networks
independent_models = []

for i, col in enumerate(targets):
    # Pipeline: Padroniza os dados (StandardScaler) e depois treina a Rede (MLP)
    model = make_pipeline(StandardScaler(), MLPRegressor(**mlp_config))
    model.fit(X_train, y_train[:, i])
    independent_models.append(model)
    print(f"Model for {col} trained successfully.")

# Model Evaluation on Unseen Test Data
joblib.dump(independent_models, 'pipeline_mlp_final.pkl')
print("\nModels saved as 'pipeline_mlp_final.pkl'")

Training with 320 samples (80%)
Model for lambda 1 trained successfully.
Model for lambda 2 trained successfully.
Model for lambda 3 trained successfully.
Model for lambda 4 trained successfully.

Models saved as 'pipeline_mlp_final.pkl'


# 4. Validation

In [6]:
print(f"VALIDATION OF THE MODEL (Tested on {len(X_test)} unseen samples)")
y_pred_test = np.zeros_like(y_test)

# Does previous code for evaluation
for i, col in enumerate(targets):
    y_pred_test[:, i] = independent_models[i].predict(X_test)
    
    r2_test = r2_score(y_test[:, i], y_pred_test[:, i])
    mse_test = mean_squared_error(y_test[:, i], y_pred_test[:, i])
    
    print(f"Results for {col}:")
    print(f"   R² : {r2_test:.4f}  |  MSE: {mse_test:.4f}\n")

VALIDATION OF THE MODEL (Tested on 80 unseen samples)
Results for lambda 1:
   R² : 0.9916  |  MSE: 2.1563

Results for lambda 2:
   R² : 0.9003  |  MSE: 0.0026

Results for lambda 3:
   R² : 0.3612  |  MSE: 0.0028

Results for lambda 4:
   R² : 0.5239  |  MSE: 0.0034



# 5. Use model

In [8]:
# Inference Pipeline Function
def predict_lambda_evolution(fck, rh, cov, time_list, trained_models):
    """
    Receives project parameters and a list of time steps.
    Returns a Numpy matrix (N_times x 4) with the values of λ1, λ2, λ3 and λ4.
    """
    results_matrix = []
    
    for t in time_list:
        # Set up the current scenario 
        current_scenario = np.array([[fck, rh, cov, t]])
        lambdas_this_year = []
        
        # Pass the scenario through the 4 experts
        for model in trained_models:
            # Get the first value from the array returned by the prediction
            prediction = model.predict(current_scenario)[0]
            lambdas_this_year.append(prediction)
            
        results_matrix.append(lambdas_this_year)
        
    return np.array(results_matrix)

# Testing the pipeline function
print("Testing the Pipeline Function")

time_analyse = [10, 20, 30, 40, 50] 
fck_proj = 30
rh_proj  = 65
cov_proj = 40

# Run the pipeline
lambda_matrix = predict_lambda_evolution(fck_proj, rh_proj, cov_proj, time_analyse, independent_models)

print(f"Results for beam with fck={fck_proj}, rh={rh_proj}%, cov={cov_proj}mm:")
print("Columns: [λ1, λ2, λ3, λ4] | Rows: Years [10, 20, 30, 40, 50]\n")
print(np.round(lambda_matrix, 4))

Testing the Pipeline Function
Results for beam with fck=30, rh=65%, cov=40mm:
Columns: [λ1, λ2, λ3, λ4] | Rows: Years [10, 20, 30, 40, 50]

[[2.98886e+01 6.14300e-01 7.82000e-02 1.11000e-01]
 [2.52779e+01 5.09600e-01 1.67000e-02 1.27000e-01]
 [2.11973e+01 4.56500e-01 4.80000e-02 1.96200e-01]
 [1.99342e+01 4.43200e-01 5.87000e-02 2.02100e-01]
 [1.93357e+01 3.99600e-01 7.68000e-02 2.51000e-01]]
